In [1]:
import pandas as pd
import torch

In [2]:
df = pd.read_csv(r'C:\Users\Nayan\Downloads\MacineLearning\Irigation_train.csv')
df.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [3]:
df = df.drop('id', axis=1)

In [4]:
df.isna().sum()

Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
Irrigation_Need            0
dtype: int64

In [5]:
cat_col = df.select_dtypes(include='object').columns
num_col = df.select_dtypes(exclude='object').columns

In [6]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
label = LabelEncoder()
scaler = StandardScaler()

In [7]:
for i in cat_col:
    df[i] = label.fit_transform(df[i])

In [8]:
y = df.Irrigation_Need
X = df.drop('Irrigation_Need', axis = 1)
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 19 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Soil_Type                630000 non-null  int64  
 1   Soil_pH                  630000 non-null  float64
 2   Soil_Moisture            630000 non-null  float64
 3   Organic_Carbon           630000 non-null  float64
 4   Electrical_Conductivity  630000 non-null  float64
 5   Temperature_C            630000 non-null  float64
 6   Humidity                 630000 non-null  float64
 7   Rainfall_mm              630000 non-null  float64
 8   Sunlight_Hours           630000 non-null  float64
 9   Wind_Speed_kmh           630000 non-null  float64
 10  Crop_Type                630000 non-null  int64  
 11  Crop_Growth_Stage        630000 non-null  int64  
 12  Season                   630000 non-null  int64  
 13  Irrigation_Type          630000 non-null  int64  
 14  Wate

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((504000, 19), (126000, 19), (504000,), (126000,))

In [10]:
Xtrain_scaled = scaler.fit_transform(X_train)
Xtest_scaled = scaler.transform(X_test)
ytrain_scaled = scaler.fit_transform(y_train.values.reshape(-1, 1))
ytest_scaled = scaler.transform(y_test.values.reshape(-1, 1))

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [12]:
Xtrain_tensor = torch.tensor(Xtrain_scaled, dtype = torch.float32)
Xtest_tensor = torch.tensor(Xtest_scaled, dtype = torch.long)
ytrain_tensor = torch.tensor(ytrain_scaled, dtype = torch.float32)
ytest_tensor = torch.tensor(ytest_scaled, dtype= torch.long)

In [13]:

train_ds = TensorDataset(Xtrain_tensor, ytrain_tensor)
test_ds = TensorDataset(Xtest_tensor, ytest_tensor)

In [14]:
train_loader = DataLoader(train_ds, shuffle=True, batch_size=32)
test_loader = DataLoader(test_ds, shuffle=False, batch_size=2)

In [35]:
class Irrigation(torch.nn.Module):
    def __init__(self,ip_dim):
        super(Irrigation, self).__init__()
        self.flat = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(ip_dim,32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32,64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,1),
          #  nn.ReLU(),
           nn.Softmax()
        )
    
    def forward(self,x):
        x = self.flat(x)
        return self.net(x)

In [36]:
ip = X_train.shape[1]
model = Irrigation(ip)

In [37]:
X_train.shape, y_train.shape

((504000, 19), (504000,))

In [38]:
y_train.unique()

array([1, 0, 2])

In [39]:
criteria = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [40]:
train_loader

In [42]:
epochs = 10

for ep in range(epochs):
    model.train()
    tot_loss = 0
    correct = 0
    total = 0

    for ip, target in train_loader:
        optimizer.zero_grad()

        op = model(ip)
        loss = criteria(op, ip)

        loss.backward()
        optimizer.step()

        tot_loss += loss.item()

        # ✅ accuracy calculation
        preds = torch.argmax(op, dim=1)
        correct += (preds == target).sum().item()
        total += target.size(0)

    avg_loss = tot_loss / len(train_loader)
    accuracy = correct / total

    print(f"Epoch [{ep+1}/{epochs}], Loss: {avg_loss:.4f}, Acc: {accuracy:.4f}")

C:\Users\Nayan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\nn\modules\module.py:1773: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


ValueError: Using a target size (torch.Size([32, 19])) that is different to the input size (torch.Size([32, 1])) is deprecated. Please ensure they have the same size.